# Flight Data Collection

This notebook is responsible for collecting flight availability
and pricing data required by the Travel Agent.

## Project Objective

The final Travel Agent must support:

### Type 1 — User provides parameters

The user provides:

- Starting location
- Destination
- Budget
- Trip duration
- Type of place/trip

The system should generate:

- Available flights
- Accommodation options
- Flight cost
- Accommodation cost
- Other estimated costs
- Total trip cost
- Complete day-by-day travel plan

### Type 2 — User provides only destination

The system should determine:

- Estimated budget
- Flight options
- Accommodation
- Accommodation cost
- Other travel requirements
- Complete tour plan

---

## Why Flight Data Is Required

The final trip cost will depend on multiple components:

Flight Cost
      +
Accommodation Cost
      +
Local / Activity Cost
      =
Total Estimated Trip Cost

Therefore, flight availability and pricing are an
important part of the Travel Agent.

---

## Flight Data Collection Flow

User / Dataset
      ↓
Origin Airport
      ↓
Destination Airport
      ↓
Departure Date
      ↓
Return Date
      ↓
Flight API
      ↓
Flight Offers
      ↓
Airline + Route + Timing + Stops + Price
      ↓
Clean Flight Dataset
      ↓
Flight Cost Calculation
      ↓
Final Travel Recommendation

---

## Important Difference from Accommodation

Accommodation can be collected around each destination.

Flights are different because flight availability and prices
depend on:

- Origin
- Destination
- Departure date
- Return date
- Number of passengers
- Cabin class

Therefore, we will first test the flight API with a controlled
origin-destination search before scaling the collection.

---

## Collection Strategy

1. Configure the flight API.
2. Test authentication.
3. Test one flight search.
4. Validate the response.
5. Design the flight dataset schema.
6. Collect flight data using controlled searches.
7. Save raw flight data.
8. Clean and validate the flight dataset.
9. Prepare flight-price features for the Travel Agent.

The flight data will later be combined with:

- Destination features
- Weather
- Places
- Accommodation
- Prices

to produce the final travel recommendations.

In [1]:


import os
import json
import time
import requests
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv



PROJECT_ROOT = Path.cwd().parent

print("Project root:")
print(PROJECT_ROOT)


# ---------------------------------------------------------
# 2. DEFINE DATA DIRECTORIES
# ---------------------------------------------------------

DATA_DIR = PROJECT_ROOT / "data"

RAW_FLIGHT_DIR = DATA_DIR / "raw" / "flights"

CLEANED_DIR = DATA_DIR / "cleaned"


# ---------------------------------------------------------
# 3. CREATE THE FLIGHT RAW-DATA DIRECTORY
# ---------------------------------------------------------
# exist_ok=True means the directory will not cause an error
# if it already exists.
# ---------------------------------------------------------

RAW_FLIGHT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CLEANED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ---------------------------------------------------------
# 4. LOAD ENVIRONMENT VARIABLES
# ---------------------------------------------------------
# This loads values stored in the project's .env file.
# We will use this later for the flight API credentials.
# ---------------------------------------------------------

load_dotenv(
    PROJECT_ROOT / ".env"
)


# ---------------------------------------------------------
# 5. DISPLAY IMPORTANT PATHS
# ---------------------------------------------------------

print("\nData directory:")
print(DATA_DIR)

print("\nRaw flight directory:")
print(RAW_FLIGHT_DIR)

print("\nCleaned data directory:")
print(CLEANED_DIR)

print("\nEnvironment file loaded.")

Project root:
c:\Users\Rushi\Desktop\Travel_Agent

Data directory:
c:\Users\Rushi\Desktop\Travel_Agent\data

Raw flight directory:
c:\Users\Rushi\Desktop\Travel_Agent\data\raw\flights

Cleaned data directory:
c:\Users\Rushi\Desktop\Travel_Agent\data\cleaned

Environment file loaded.


In [2]:
# Load the 50 destinations prepared in Notebook 01

destination_master = pd.read_csv(
    DATA_DIR / "raw" / "destination_master.csv"
)

print("Rows:", len(destination_master))
print("Columns:", destination_master.columns.tolist())

display(destination_master.head())

Rows: 50
Columns: ['destination_id', 'destination', 'search_query']


,destination_id,destination,search_query
0,1,Goa,Goa
1,2,Munnar,Munnar
2,3,Manali,Manali
3,4,Jaipur,Jaipur
4,5,Udaipur,Udaipur


In [3]:
# Load the validated destination location data

destination_locations = pd.read_csv(
    DATA_DIR / "raw" / "places" / "destination_locations.csv"
)

print("Rows:", len(destination_locations))
print("Columns:", destination_locations.columns.tolist())

display(
    destination_locations[
        ["destination", "latitude", "longitude"]
    ].head()
)

Rows: 50
Columns: ['destination_id', 'destination', 'search_query', 'resolved_name', 'country', 'country_code', 'state', 'state_code', 'latitude', 'longitude', 'result_type', 'formatted_address', 'confidence', 'match_type', 'place_id']


,destination,latitude,longitude
0,Goa,15.300454,74.085513
1,Munnar,10.086996,77.060091
2,Manali,32.245461,77.187293
3,Jaipur,26.915458,75.818982
4,Udaipur,24.578721,73.686257


# Airport Mapping

Flights require airport codes, not just destination names.

For each destination, we need to identify the most suitable
airport for flight searches.

Flow:

Destination
    ↓
Coordinates
    ↓
Nearest / suitable airport
    ↓
IATA airport code
    ↓
Flight API search

In [4]:
# Map each destination to its main airport code.

airport_mapping = {
    "Goa": "GOI",
    "Munnar": "COK",
    "Manali": "KUU",
    "Jaipur": "JAI",
    "Udaipur": "UDR",
    "Jaisalmer": "JSA",
    "Jodhpur": "JDH",
    "Agra": "AGR",
    "Varanasi": "VNS",
    "Rishikesh": "DED",
    "Shimla": "SLV",
    "Mussoorie": "DED",
    "Nainital": "DED",
    "Darjeeling": "IXB",
    "Gangtok": "IXB",
    "Ooty": "CJB",
    "Kodaikanal": "CJB",
    "Coorg": "IXE",
    "Wayanad": "CCJ",
    "Alappuzha": "COK",
    "Kochi": "COK",
    "Thiruvananthapuram": "TRV",
    "Varkala": "TRV",
    "Pondicherry": "MAA",
    "Mahabalipuram": "MAA",
    "Hampi": "VDY",
    "Mysore": "MYQ",
    "Gokarna": "GOI",
    "Andaman": "IXZ",
    "Mumbai": "BOM",
    "Delhi": "DEL",
    "Amritsar": "ATQ",
    "Ladakh": "IXL",
    "Srinagar": "SXR",
    "Dharamshala": "DHM",
    "Kolkata": "CCU",
    "Bengaluru": "BLR",
    "Hyderabad": "HYD",
    "Chennai": "MAA",
    "Pune": "PNQ",
    "Ahmedabad": "AMD",
    "Bhopal": "BHO",
    "Indore": "IDR",
    "Ranchi": "IXR",
    "Bhubaneswar": "BBI",
    "Shillong": "SHL",
    "Kaziranga": "JRH",
    "Jim Corbett": "DED",
    "Ranthambore": "JAI",
    "Pahalgam": "SXR"
}

# Add the airport code to our destination dataset.
destination_master["airport_code"] = (
    destination_master["destination"]
    .map(airport_mapping)
)

print("Rows:", len(destination_master))
print(
    "Missing airport codes:",
    destination_master["airport_code"].isna().sum()
)

display(
    destination_master[
        ["destination", "airport_code"]
    ].head(10)
)

Rows: 50
Missing airport codes: 0


,destination,airport_code
0,Goa,GOI
1,Munnar,COK
2,Manali,KUU
3,Jaipur,JAI
4,Udaipur,UDR
5,Jaisalmer,JSA
6,Jodhpur,JDH
7,Agra,AGR
8,Varanasi,VNS
9,Rishikesh,DED


# Flight API Authentication

We will use the Amadeus API for flight searches.

First, we verify that the API credentials are loaded
correctly. We will not search for flights yet.

In [5]:
# Load the SerpApi key from .env

SERPAPI_API_KEY = os.getenv("SERPAPI_API_KEY")

print(
    "SerpApi API key available:",
    bool(SERPAPI_API_KEY)
)

SerpApi API key available: True


# Test Flight Search

Before collecting flight data, we will test one route.

Test route:

Hyderabad (HYD) → Goa (GOI)

The test will verify that:
- SerpApi authentication works
- Google Flights search works
- Flight offers are returned
- Prices and flight details are available

Only one API search will be made.

In [6]:
# Test one round-trip flight search

flight_test_params = {
    "engine": "google_flights",
    "departure_id": "HYD",
    "arrival_id": "GOI",
    "outbound_date": "2026-09-15",
    "return_date": "2026-09-18",
    "adults": 1,
    "travel_class": 1,       # Economy
    "currency": "INR",
    "hl": "en",
    "gl": "in",
    "api_key": SERPAPI_API_KEY
}

response = requests.get(
    "https://serpapi.com/search",
    params=flight_test_params,
    timeout=60
)

print("Status code:", response.status_code)

flight_test_result = response.json()

print("Search status:")
print(
    flight_test_result.get(
        "search_metadata",
        {}
    ).get("status")
)

Status code: 200
Search status:
Success


In [7]:
# Inspect the main sections returned by SerpApi

print("Response sections:")
print(list(flight_test_result.keys()))

Response sections:
['search_metadata', 'search_parameters', 'best_flights', 'other_flights', 'price_insights', 'airports']


In [8]:
# Inspect the first flight offer returned by Google Flights

if flight_test_result.get("best_flights"):
    first_offer = flight_test_result["best_flights"][0]

    print("Fields in flight offer:")
    print(first_offer.keys())

    print("\nFirst flight offer:")
    display(first_offer)

else:
    print("No best flight offers returned.")

Fields in flight offer:
dict_keys(['flights', 'total_duration', 'carbon_emissions', 'price', 'type', 'airline_logo', 'departure_token'])

First flight offer:


{'flights': [{'departure_airport': {'name': 'Rajiv Gandhi International Airport',
    'id': 'HYD',
    'time': '2026-09-15 11:20'},
   'arrival_airport': {'name': 'Goa Dabolim International Airport',
    'id': 'GOI',
    'time': '2026-09-15 12:35'},
   'duration': 75,
   'airplane': 'Airbus A320neo',
   'airline': 'IndiGo',
   'airline_logo': 'https://www.gstatic.com/flights/airline_logos/70px/6E.png',
   'travel_class': 'Economy',
   'flight_number': '6E 362',
   'legroom': '28 in',
   'extensions': ['Below average legroom (28 in)',
    'Carbon emissions estimate: 55 kg']}],
 'total_duration': 75,
 'carbon_emissions': {'this_flight': 56000,
  'typical_for_this_route': 51000,
  'difference_percent': 10},
 'price': 9538,
 'type': 'Round trip',
 'airline_logo': 'https://www.gstatic.com/flights/airline_logos/70px/6E.png',
 'departure_token': 'WyJDalJJUzNoME0ycFNYM00yYTFGQlEydDJaWGRDUnkwdExTMHRMUzF2YTNoeE9DMXVNa0ZCUVVGQlIzRlFkbFZ2UmxoVWNtZEJFZ1UyUlRNMk1ob0tDTUpLRUFBYUEwbE9VamdjY0k1TyIsW1si

In [9]:
# Check how many legs are in the first round-trip offer

legs = first_offer["flights"]

print("Number of flight legs:", len(legs))

for i, leg in enumerate(legs, start=1):
    print(f"\nLeg {i}")
    print("Airline:", leg["airline"])
    print("Flight:", leg["flight_number"])
    print("From:", leg["departure_airport"]["id"])
    print("To:", leg["arrival_airport"]["id"])
    print("Departure:", leg["departure_airport"]["time"])
    print("Arrival:", leg["arrival_airport"]["time"])
    print("Duration:", leg["duration"], "minutes")

Number of flight legs: 1

Leg 1
Airline: IndiGo
Flight: 6E 362
From: HYD
To: GOI
Departure: 2026-09-15 11:20
Arrival: 2026-09-15 12:35
Duration: 75 minutes


In [10]:
# Check all keys and important values in the first offer

print("Offer price:", first_offer.get("price"))
print("Offer type:", first_offer.get("type"))
print("Total duration:", first_offer.get("total_duration"))

print("\nNumber of flight segments:",
      len(first_offer.get("flights", [])))

print("\nPrice insights:")
display(flight_test_result.get("price_insights", {}))

Offer price: 9538
Offer type: Round trip
Total duration: 75

Number of flight segments: 1

Price insights:


{'lowest_price': 9538,
 'price_level': 'typical',
 'typical_price_range': [7000, 11500],
 'price_history': [[1782498600, 8435],
  [1782585000, 8435],
  [1782671400, 8435],
  [1782757800, 8435],
  [1782844200, 8435],
  [1782930600, 8435],
  [1783017000, 8435],
  [1783103400, 8435],
  [1783189800, 8435],
  [1783276200, 8297],
  [1783362600, 8297],
  [1783449000, 8297],
  [1783535400, 7595],
  [1783621800, 7595],
  [1783708200, 7595],
  [1783794600, 7595],
  [1783881000, 7595],
  [1783967400, 7595],
  [1784053800, 7385],
  [1784140200, 7385],
  [1784226600, 7385],
  [1784313000, 7385],
  [1784399400, 7385],
  [1784485800, 7385],
  [1784572200, 8435],
  [1784658600, 8772],
  [1784745000, 8772],
  [1784831400, 8297],
  [1784917800, 8297],
  [1785004200, 8297],
  [1785090600, 8297],
  [1785177000, 8297],
  [1785263400, 8523],
  [1785349800, 8772],
  [1785436200, 8523],
  [1785522600, 8187],
  [1785609000, 8187],
  [1785695400, 8371],
  [1785781800, 8371],
  [1785868200, 7948],
  [1785954600,

In [11]:
# View the complete first offer in a readable format

print(
    json.dumps(
        first_offer,
        indent=2
    )
)

{
  "flights": [
    {
      "departure_airport": {
        "name": "Rajiv Gandhi International Airport",
        "id": "HYD",
        "time": "2026-09-15 11:20"
      },
      "arrival_airport": {
        "name": "Goa Dabolim International Airport",
        "id": "GOI",
        "time": "2026-09-15 12:35"
      },
      "duration": 75,
      "airplane": "Airbus A320neo",
      "airline": "IndiGo",
      "airline_logo": "https://www.gstatic.com/flights/airline_logos/70px/6E.png",
      "travel_class": "Economy",
      "flight_number": "6E 362",
      "legroom": "28 in",
      "extensions": [
        "Below average legroom (28 in)",
        "Carbon emissions estimate: 55 kg"
      ]
    }
  ],
  "total_duration": 75,
  "carbon_emissions": {
    "this_flight": 56000,
    "typical_for_this_route": 51000,
    "difference_percent": 10
  },
  "price": 9538,
  "type": "Round trip",
  "airline_logo": "https://www.gstatic.com/flights/airline_logos/70px/6E.png",
  "departure_token": "WyJDalJJUzNo

In [12]:
# Get the departure token from our selected outbound flight

departure_token = first_offer["departure_token"]

return_params = {
    "engine": "google_flights",
    "departure_id": "HYD",
    "arrival_id": "GOI",
    "outbound_date": "2026-09-15",
    "return_date": "2026-09-18",
    "departure_token": departure_token,
    "currency": "INR",
    "hl": "en",
    "gl": "in",
    "api_key": SERPAPI_API_KEY
}

return_response = requests.get(
    "https://serpapi.com/search",
    params=return_params,
    timeout=60
)

print("Status code:", return_response.status_code)

return_result = return_response.json()

print(
    "Search status:",
    return_result.get("search_metadata", {}).get("status")
)

Status code: 200
Search status: Success


In [13]:
# Check the sections returned for the return-flight search

print("Response sections:")
print(list(return_result.keys()))

print("\nBest flights returned:",
      len(return_result.get("best_flights", [])))

print("Other flights returned:",
      len(return_result.get("other_flights", [])))

Response sections:
['search_metadata', 'search_parameters', 'other_flights', 'airports']

Best flights returned: 0
Other flights returned: 4


In [14]:
# Inspect the first return-flight option

return_offer = return_result["other_flights"][0]

print("Fields:")
print(return_offer.keys())

print("\nReturn flight offer:")
display(return_offer)

Fields:
dict_keys(['flights', 'total_duration', 'carbon_emissions', 'price', 'type', 'airline_logo', 'booking_token'])

Return flight offer:


{'flights': [{'departure_airport': {'name': 'Goa Dabolim International Airport',
    'id': 'GOI',
    'time': '2026-09-18 19:05'},
   'arrival_airport': {'name': 'Rajiv Gandhi International Airport',
    'id': 'HYD',
    'time': '2026-09-18 20:25'},
   'duration': 80,
   'airplane': 'Airbus A320neo',
   'airline': 'IndiGo',
   'airline_logo': 'https://www.gstatic.com/flights/airline_logos/70px/6E.png',
   'travel_class': 'Economy',
   'flight_number': '6E 117',
   'legroom': '28 in',
   'extensions': ['Below average legroom (28 in)']}],
 'total_duration': 80,
 'carbon_emissions': {'typical_for_this_route': 51000},
 'price': 9538,
 'type': 'Round trip',
 'airline_logo': 'https://www.gstatic.com/flights/airline_logos/70px/6E.png',
 'booking_token': 'WyJDalJJVTJ0NE9GbGxhMkZSTURoQlEzQkllVUZDUnkwdExTMHRMUzB0TFc5cmFHRXlNVUZCUVVGQlIzRlFkMGh2Um5CVVN6QkJFZ1UyUlRFeE54b0tDTUpLRUFBYUEwbE9VamdjY0k1TyIsW1siSFlEIiwiMjAyNi0wOS0xNSIsIkdPSSIsbnVsbCwiNkUiLCIzNjIiXV0sW1siR09JIiwiMjAyNi0wOS0xOCIsIkhZRCIsbn

# Flight Dataset Structure

Each row represents one complete round-trip flight option.

We keep:
- Trip information
- Outbound flight details
- Return flight details
- Total price
- Duration
- Stops
- Travel class

This structure will later allow us to compare flight options
and calculate the total travel cost.

In [16]:
# Define the columns for our flight dataset

flight_columns = [
    "origin",
    "destination",
    "departure_date",
    "return_date",

    "outbound_airline",
    "outbound_flight_number",
    "outbound_departure",
    "outbound_arrival",
    "outbound_duration",
    "outbound_stops",

    "return_airline",
    "return_flight_number",
    "return_departure",
    "return_arrival",
    "return_duration",
    "return_stops",

    "total_duration",
    "price",
    "currency",
    "travel_class"
]

flights_df = pd.DataFrame(columns=flight_columns)

print("Columns:", len(flights_df.columns))
print(flights_df.columns.tolist())

Columns: 20
['origin', 'destination', 'departure_date', 'return_date', 'outbound_airline', 'outbound_flight_number', 'outbound_departure', 'outbound_arrival', 'outbound_duration', 'outbound_stops', 'return_airline', 'return_flight_number', 'return_departure', 'return_arrival', 'return_duration', 'return_stops', 'total_duration', 'price', 'currency', 'travel_class']


In [17]:
# Extract outbound flight details from our test offer

outbound = first_offer["flights"][0]

outbound_data = {
    "origin": outbound["departure_airport"]["id"],
    "destination": outbound["arrival_airport"]["id"],
    "departure_date": "2026-09-15",
    "return_date": "2026-09-18",

    "outbound_airline": outbound["airline"],
    "outbound_flight_number": outbound["flight_number"],
    "outbound_departure": outbound["departure_airport"]["time"],
    "outbound_arrival": outbound["arrival_airport"]["time"],
    "outbound_duration": outbound["duration"],
    
    "travel_class": outbound["travel_class"]
}

print(outbound_data)

{'origin': 'HYD', 'destination': 'GOI', 'departure_date': '2026-09-15', 'return_date': '2026-09-18', 'outbound_airline': 'IndiGo', 'outbound_flight_number': '6E 362', 'outbound_departure': '2026-09-15 11:20', 'outbound_arrival': '2026-09-15 12:35', 'outbound_duration': 75, 'travel_class': 'Economy'}


In [18]:
# Extract return flight details

return_flight = return_offer["flights"][0]

return_data = {
    "return_airline": return_flight["airline"],
    "return_flight_number": return_flight["flight_number"],
    "return_departure": return_flight["departure_airport"]["time"],
    "return_arrival": return_flight["arrival_airport"]["time"],
    "return_duration": return_flight["duration"]
}

print(return_data)

{'return_airline': 'IndiGo', 'return_flight_number': '6E 117', 'return_departure': '2026-09-18 19:05', 'return_arrival': '2026-09-18 20:25', 'return_duration': 80}


In [19]:
# Combine outbound, return, and common trip details

flight_record = {
    **outbound_data,
    **return_data,

    "outbound_stops": len(outbound.get("extensions", [])),
    "return_stops": len(return_flight.get("extensions", [])),

    "total_duration": (
        first_offer["total_duration"]
        + return_offer["total_duration"]
    ),

    "price": first_offer["price"],
    "currency": "INR",
}

print(flight_record)

{'origin': 'HYD', 'destination': 'GOI', 'departure_date': '2026-09-15', 'return_date': '2026-09-18', 'outbound_airline': 'IndiGo', 'outbound_flight_number': '6E 362', 'outbound_departure': '2026-09-15 11:20', 'outbound_arrival': '2026-09-15 12:35', 'outbound_duration': 75, 'travel_class': 'Economy', 'return_airline': 'IndiGo', 'return_flight_number': '6E 117', 'return_departure': '2026-09-18 19:05', 'return_arrival': '2026-09-18 20:25', 'return_duration': 80, 'outbound_stops': 2, 'return_stops': 1, 'total_duration': 155, 'price': 9538, 'currency': 'INR'}


In [20]:
# Calculate stops correctly from the number of flight segments

outbound_stops = max(len(first_offer["flights"]) - 1, 0)
return_stops = max(len(return_offer["flights"]) - 1, 0)

print("Outbound stops:", outbound_stops)
print("Return stops:", return_stops)

Outbound stops: 0
Return stops: 0


In [21]:
# Create one complete round-trip flight record

flight_record = {
    **outbound_data,
    **return_data,

    "outbound_stops": outbound_stops,
    "return_stops": return_stops,

    "total_duration": (
        first_offer["total_duration"]
        + return_offer["total_duration"]
    ),

    "price": first_offer["price"],
    "currency": "INR"
}

# Convert the record into a one-row dataframe
test_flight_df = pd.DataFrame([flight_record])

display(test_flight_df)

,origin,destination,departure_date,return_date,outbound_airline,outbound_flight_number,outbound_departure,outbound_arrival,outbound_duration,travel_class,return_airline,return_flight_number,return_departure,return_arrival,return_duration,outbound_stops,return_stops,total_duration,price,currency
0,HYD,GOI,2026-09-15,2026-09-18,IndiGo,6E 362,2026-09-15 11:20,2026-09-15 12:35,75,Economy,IndiGo,6E 117,2026-09-18 19:05,2026-09-18 20:25,80,0,0,155,9538,INR


In [22]:
def extract_flight_offer(offer, origin, destination, departure_date, return_date):
    """
    Convert one SerpApi flight offer into our standard dataset format.
    """

    # Get the flight segments
    segments = offer.get("flights", [])

    if not segments:
        return None

    # First and last segments give the overall outbound journey
    first_segment = segments[0]
    last_segment = segments[-1]

    # Calculate stops from number of segments
    outbound_stops = max(len(segments) - 1, 0)

    record = {
        "origin": origin,
        "destination": destination,
        "departure_date": departure_date,
        "return_date": return_date,

        "outbound_airline": first_segment.get("airline"),
        "outbound_flight_number": first_segment.get("flight_number"),
        "outbound_departure": first_segment["departure_airport"].get("time"),
        "outbound_arrival": last_segment["arrival_airport"].get("time"),
        "outbound_duration": offer.get("total_duration"),
        "outbound_stops": outbound_stops,

        "return_airline": None,
        "return_flight_number": None,
        "return_departure": None,
        "return_arrival": None,
        "return_duration": None,
        "return_stops": None,

        "total_duration": None,
        "price": offer.get("price"),
        "currency": "INR",
        "travel_class": first_segment.get("travel_class")
    }

    return record

In [23]:
def create_round_trip_record(
    outbound_offer,
    return_offer,
    origin,
    destination,
    departure_date,
    return_date
):
    """Combine outbound and return offers into one flight record."""

    # Get flight segments
    outbound_segments = outbound_offer.get("flights", [])
    return_segments = return_offer.get("flights", [])

    if not outbound_segments or not return_segments:
        return None

    # First and last segments describe the complete journey
    outbound_first = outbound_segments[0]
    outbound_last = outbound_segments[-1]

    return_first = return_segments[0]
    return_last = return_segments[-1]

    # Number of stops = number of segments - 1
    outbound_stops = max(len(outbound_segments) - 1, 0)
    return_stops = max(len(return_segments) - 1, 0)

    return {
        "origin": origin,
        "destination": destination,
        "departure_date": departure_date,
        "return_date": return_date,

        # Outbound flight
        "outbound_airline": outbound_first.get("airline"),
        "outbound_flight_number": outbound_first.get("flight_number"),
        "outbound_departure": outbound_first["departure_airport"].get("time"),
        "outbound_arrival": outbound_last["arrival_airport"].get("time"),
        "outbound_duration": outbound_offer.get("total_duration"),
        "outbound_stops": outbound_stops,

        # Return flight
        "return_airline": return_first.get("airline"),
        "return_flight_number": return_first.get("flight_number"),
        "return_departure": return_first["departure_airport"].get("time"),
        "return_arrival": return_last["arrival_airport"].get("time"),
        "return_duration": return_offer.get("total_duration"),
        "return_stops": return_stops,

        # Overall trip information
        "total_duration": (
            outbound_offer.get("total_duration", 0)
            + return_offer.get("total_duration", 0)
        ),
        "price": outbound_offer.get("price"),
        "currency": "INR",
        "travel_class": outbound_first.get("travel_class")
    }

In [24]:
# Create our first complete flight record using the function

test_record = create_round_trip_record(
    first_offer,
    return_offer,
    "HYD",
    "GOI",
    "2026-09-15",
    "2026-09-18"
)

test_flight_df = pd.DataFrame([test_record])

display(test_flight_df)

,origin,destination,departure_date,return_date,outbound_airline,outbound_flight_number,outbound_departure,outbound_arrival,outbound_duration,outbound_stops,return_airline,return_flight_number,return_departure,return_arrival,return_duration,return_stops,total_duration,price,currency,travel_class
0,HYD,GOI,2026-09-15,2026-09-18,IndiGo,6E 362,2026-09-15 11:20,2026-09-15 12:35,75,0,IndiGo,6E 117,2026-09-18 19:05,2026-09-18 20:25,80,0,155,9538,INR,Economy


In [25]:
# Select a small sample of destinations for flight-data collection

sample_destinations = [
    "Goa",
    "Munnar",
    "Manali",
    "Jaipur",
    "Udaipur",
    "Delhi",
    "Mumbai",
    "Kochi",
    "Srinagar",
    "Bengaluru"
]

print("Sample destinations:", len(sample_destinations))
print(sample_destinations)

Sample destinations: 10
['Goa', 'Munnar', 'Manali', 'Jaipur', 'Udaipur', 'Delhi', 'Mumbai', 'Kochi', 'Srinagar', 'Bengaluru']


In [31]:
# Select airport codes for our sample destinations

sample_airports = destination_master[
    destination_master["destination"].isin(sample_destinations)
][["destination", "airport_code"]].copy()

display(sample_airports)

,destination,airport_code
0,Goa,GOI
1,Munnar,COK
2,Manali,KUU
3,Jaipur,JAI
4,Udaipur,UDR
20,Kochi,COK
29,Mumbai,BOM
30,Delhi,DEL
33,Srinagar,SXR
36,Bengaluru,BLR


### This function is for searching single flight either outbound or return

In [32]:
def search_flights(origin, destination, departure_date, return_date):
    """
    Search round-trip flights for one destination.
    Returns a list of complete flight records.
    """

    # Find outbound flights
    params = {
        "engine": "google_flights",
        "departure_id": origin,
        "arrival_id": destination,
        "outbound_date": departure_date,
        "return_date": return_date,
        "adults": 1,
        "travel_class": 1,
        "currency": "INR",
        "hl": "en",
        "gl": "in",
        "api_key": SERPAPI_API_KEY
    }

    response = requests.get(
        "https://serpapi.com/search",
        params=params,
        timeout=60
    )

    response.raise_for_status()

    result = response.json()

    # Get available outbound offers
    outbound_offers = (
        result.get("best_flights", [])
        + result.get("other_flights", [])
    )

    print(
        f"Outbound options found: {len(outbound_offers)}"
    )

    return result

### This function will return both outbound and return flights if exists

In [33]:
def search_round_trip(origin, destination, departure_date, return_date):
    """
    Search outbound and return flights and combine them into records.
    """

    # ---------- OUTBOUND SEARCH ----------
    outbound_params = {
        "engine": "google_flights",
        "departure_id": origin,
        "arrival_id": destination,
        "outbound_date": departure_date,
        "return_date": return_date,
        "adults": 1,
        "travel_class": 1,
        "currency": "INR",
        "hl": "en",
        "gl": "in",
        "api_key": SERPAPI_API_KEY
    }

    response = requests.get(
        "https://serpapi.com/search",
        params=outbound_params,
        timeout=60
    )

    response.raise_for_status()
    outbound_result = response.json()

    # Combine both possible result sections
    outbound_offers = (
        outbound_result.get("best_flights", [])
        + outbound_result.get("other_flights", [])
    )

    if not outbound_offers:
        print("No outbound flights found.")
        return []

    # Use the first outbound offer to obtain a return-flight search token
    outbound_offer = outbound_offers[0]

    departure_token = outbound_offer.get("departure_token")

    if not departure_token:
        print("No departure token found.")
        return []

    # ---------- RETURN SEARCH ----------
    return_params = {
        "engine": "google_flights",
        "departure_id": origin,
        "arrival_id": destination,
        "outbound_date": departure_date,
        "return_date": return_date,
        "departure_token": departure_token,
        "currency": "INR",
        "hl": "en",
        "gl": "in",
        "api_key": SERPAPI_API_KEY
    }

    return_response = requests.get(
        "https://serpapi.com/search",
        params=return_params,
        timeout=60
    )

    return_response.raise_for_status()
    return_result = return_response.json()

    # Return options may appear in either section
    return_offers = (
        return_result.get("best_flights", [])
        + return_result.get("other_flights", [])
    )

    if not return_offers:
        print("No return flights found.")
        return []

    # Create complete round-trip records
    records = []

    for return_offer in return_offers:
        record = create_round_trip_record(
            outbound_offer,
            return_offer,
            origin,
            destination,
            departure_date,
            return_date
        )

        if record:
            records.append(record)

    return records

In [34]:
# Test the reusable round-trip search function with Goa

goa_records = search_round_trip(
    origin="HYD",
    destination="GOI",
    departure_date="2026-09-15",
    return_date="2026-09-18"
)

print("Complete flight records:", len(goa_records))

Complete flight records: 4


In [35]:
# Convert the collected records into a dataframe

goa_flights_df = pd.DataFrame(goa_records)

print("Rows:", len(goa_flights_df))
print("Columns:", goa_flights_df.columns.tolist())

display(goa_flights_df)

Rows: 4
Columns: ['origin', 'destination', 'departure_date', 'return_date', 'outbound_airline', 'outbound_flight_number', 'outbound_departure', 'outbound_arrival', 'outbound_duration', 'outbound_stops', 'return_airline', 'return_flight_number', 'return_departure', 'return_arrival', 'return_duration', 'return_stops', 'total_duration', 'price', 'currency', 'travel_class']


,origin,destination,departure_date,return_date,outbound_airline,outbound_flight_number,outbound_departure,outbound_arrival,outbound_duration,outbound_stops,return_airline,return_flight_number,return_departure,return_arrival,return_duration,return_stops,total_duration,price,currency,travel_class
0,HYD,GOI,2026-09-15,2026-09-18,IndiGo,6E 362,2026-09-15 11:20,2026-09-15 12:35,75,0,IndiGo,6E 117,2026-09-18 19:05,2026-09-18 20:25,80,0,155,9538,INR,Economy
1,HYD,GOI,2026-09-15,2026-09-18,IndiGo,6E 362,2026-09-15 11:20,2026-09-15 12:35,75,0,IndiGo,6E 985,2026-09-18 21:40,2026-09-18 22:50,70,0,145,9538,INR,Economy
2,HYD,GOI,2026-09-15,2026-09-18,IndiGo,6E 362,2026-09-15 11:20,2026-09-15 12:35,75,0,IndiGo,6E 374,2026-09-18 16:35,2026-09-18 17:50,75,0,150,9538,INR,Economy
3,HYD,GOI,2026-09-15,2026-09-18,IndiGo,6E 362,2026-09-15 11:20,2026-09-15 12:35,75,0,IndiGo,6E 6177,2026-09-18 20:05,2026-09-18 21:15,70,0,145,9538,INR,Economy


In [36]:
# Check for duplicate flight options

duplicate_count = goa_flights_df.duplicated(
    subset=[
        "outbound_flight_number",
        "outbound_departure",
        "return_flight_number",
        "return_departure"
    ]
).sum()

print("Duplicate flight options:", duplicate_count)

Duplicate flight options: 0


In [37]:
# Create the master dataframe for all flight records

flights_master = pd.DataFrame(
    goa_records,
    columns=flight_columns
)

print("Flight records:", len(flights_master))
print("Columns:", len(flights_master.columns))

display(flights_master)

Flight records: 4
Columns: 20


,origin,destination,departure_date,return_date,outbound_airline,outbound_flight_number,outbound_departure,outbound_arrival,outbound_duration,outbound_stops,return_airline,return_flight_number,return_departure,return_arrival,return_duration,return_stops,total_duration,price,currency,travel_class
0,HYD,GOI,2026-09-15,2026-09-18,IndiGo,6E 362,2026-09-15 11:20,2026-09-15 12:35,75,0,IndiGo,6E 117,2026-09-18 19:05,2026-09-18 20:25,80,0,155,9538,INR,Economy
1,HYD,GOI,2026-09-15,2026-09-18,IndiGo,6E 362,2026-09-15 11:20,2026-09-15 12:35,75,0,IndiGo,6E 985,2026-09-18 21:40,2026-09-18 22:50,70,0,145,9538,INR,Economy
2,HYD,GOI,2026-09-15,2026-09-18,IndiGo,6E 362,2026-09-15 11:20,2026-09-15 12:35,75,0,IndiGo,6E 374,2026-09-18 16:35,2026-09-18 17:50,75,0,150,9538,INR,Economy
3,HYD,GOI,2026-09-15,2026-09-18,IndiGo,6E 362,2026-09-15 11:20,2026-09-15 12:35,75,0,IndiGo,6E 6177,2026-09-18 20:05,2026-09-18 21:15,70,0,145,9538,INR,Economy


In [38]:
# Check important values and duplicate flight options

print("Missing values:")
print(
    flights_master[
        ["origin", "destination", "price",
         "outbound_flight_number", "return_flight_number"]
    ].isna().sum()
)

print("\nDuplicate records:",
      flights_master.duplicated().sum())

Missing values:
origin                    0
destination               0
price                     0
outbound_flight_number    0
return_flight_number      0
dtype: int64

Duplicate records: 0


In [40]:
# Set fixed dates for our sample flight dataset

DEPARTURE_DATE = "2026-09-15"
RETURN_DATE = "2026-09-18"

print("Departure date:", DEPARTURE_DATE)
print("Return date:", RETURN_DATE)

Departure date: 2026-09-15
Return date: 2026-09-18


In [41]:
# Collect round-trip flight options for Munnar

munnar_records = search_round_trip(
    origin="HYD",
    destination="COK",
    departure_date=DEPARTURE_DATE,
    return_date=RETURN_DATE
)

print("Complete flight records:", len(munnar_records))

Complete flight records: 9


In [42]:
# Add Munnar flight records to the master dataframe

munnar_flights_df = pd.DataFrame(
    munnar_records,
    columns=flights_master.columns
)

flights_master = pd.concat(
    [flights_master, munnar_flights_df],
    ignore_index=True
)

print("Total flight records:", len(flights_master))
print("Destinations:", flights_master["destination"].nunique())

display(
    flights_master.groupby("destination").size()
)

Total flight records: 13
Destinations: 2


destination
COK    9
GOI    4
dtype: int64

In [44]:
munnar_flights_df.head()

,origin,destination,departure_date,return_date,outbound_airline,outbound_flight_number,outbound_departure,outbound_arrival,outbound_duration,outbound_stops,return_airline,return_flight_number,return_departure,return_arrival,return_duration,return_stops,total_duration,price,currency,travel_class
0,HYD,COK,2026-09-15,2026-09-18,IndiGo,6E 6417,2026-09-15 17:45,2026-09-15 21:30,225,1,IndiGo,6E 304,2026-09-18 12:05,2026-09-18 13:40,95,0,320,14132,INR,Economy
1,HYD,COK,2026-09-15,2026-09-18,IndiGo,6E 6417,2026-09-15 17:45,2026-09-15 21:30,225,1,IndiGo,6E 6235,2026-09-18 13:50,2026-09-18 15:25,95,0,320,14132,INR,Economy
2,HYD,COK,2026-09-15,2026-09-18,IndiGo,6E 6417,2026-09-15 17:45,2026-09-15 21:30,225,1,IndiGo,6E 6011,2026-09-18 13:55,2026-09-18 17:25,210,1,435,14132,INR,Economy
3,HYD,COK,2026-09-15,2026-09-18,IndiGo,6E 6417,2026-09-15 17:45,2026-09-15 21:30,225,1,IndiGo,6E 6535,2026-09-18 09:40,2026-09-18 17:45,485,1,710,14132,INR,Economy
4,HYD,COK,2026-09-15,2026-09-18,IndiGo,6E 6417,2026-09-15 17:45,2026-09-15 21:30,225,1,IndiGo,6E 196,2026-09-18 12:00,2026-09-18 21:00,540,1,765,14132,INR,Economy


In [43]:
# Collect flight data for multiple destinations

batch_destinations = [
    "Manali",
    "Jaipur",
    "Udaipur",
    "Delhi",
    "Mumbai"
]

batch_records = []

for destination in batch_destinations:

    # Get airport code from our destination master
    airport_code = destination_master.loc[
        destination_master["destination"] == destination,
        "airport_code"
    ].iloc[0]

    print("=" * 50)
    print(f"Collecting flights: {destination} ({airport_code})")

    try:
        # Each destination uses 2 API requests
        records = search_round_trip(
            origin="HYD",
            destination=airport_code,
            departure_date=DEPARTURE_DATE,
            return_date=RETURN_DATE
        )

        batch_records.extend(records)

        print(f"Success → {len(records)} flight records")

    except Exception as e:
        print(f"FAILED → {destination}")
        print(f"Reason: {e}")

print("\nBatch collection completed.")
print("Total records collected:", len(batch_records))

No outbound flights found.
Success → 0 flight records
Success → 3 flight records
Success → 3 flight records
Success → 29 flight records
Success → 17 flight records

Batch collection completed.
Total records collected: 52


In [45]:
# Add the newly collected batch to the master flight dataset

batch_df = pd.DataFrame(
    batch_records,
    columns=flights_master.columns
)

flights_master = pd.concat(
    [flights_master, batch_df],
    ignore_index=True
)

print("Total flight records:", len(flights_master))
print("Unique destinations:", flights_master["destination"].nunique())

display(
    flights_master.groupby("destination").size()
)

Total flight records: 65
Unique destinations: 6


destination
BOM    17
COK     9
DEL    29
GOI     4
JAI     3
UDR     3
dtype: int64

In [46]:
# Collect another batch of flight data
# Goal: increase the dataset from 65 records toward 100+

next_destinations = [
    "Manali",
    "Kochi",
    "Srinagar",
    "Bengaluru"
]

next_batch_records = []

for destination in next_destinations:

    # Get the airport code from our master destination data
    airport_code = destination_master.loc[
        destination_master["destination"] == destination,
        "airport_code"
    ].iloc[0]

    print("=" * 50)
    print(f"Collecting flights: {destination} ({airport_code})")

    try:
        records = search_round_trip(
            origin="HYD",
            destination=airport_code,
            departure_date=DEPARTURE_DATE,
            return_date=RETURN_DATE
        )

        next_batch_records.extend(records)

        print(f"Success → {len(records)} flight records")

    except Exception as e:
        print(f"FAILED → {destination}")
        print(f"Reason: {e}")

print("\nBatch collection completed.")
print("New records:", len(next_batch_records))

No outbound flights found.
Success → 0 flight records
Success → 9 flight records
Success → 3 flight records
Success → 13 flight records

Batch collection completed.
New records: 25


In [47]:
# Add the latest flight records to the master dataset

next_batch_df = pd.DataFrame(
    next_batch_records,
    columns=flights_master.columns
)

flights_master = pd.concat(
    [flights_master, next_batch_df],
    ignore_index=True
)

print("Total flight records:", len(flights_master))
print("Unique destinations:", flights_master["destination"].nunique())

display(
    flights_master.groupby("destination").size()
)

Total flight records: 90
Unique destinations: 8


destination
BLR    13
BOM    17
COK    18
DEL    29
GOI     4
JAI     3
SXR     3
UDR     3
dtype: int64

In [48]:
# Final validation of the flight dataset

print("Total records:", len(flights_master))
print("Unique destinations:", flights_master["destination"].nunique())

print("\nMissing values:")
print(flights_master.isna().sum())

print("\nDuplicate records:", flights_master.duplicated().sum())

print("\nRecords per destination:")
print(flights_master.groupby("destination").size())

Total records: 90
Unique destinations: 8

Missing values:
origin                    0
destination               0
departure_date            0
return_date               0
outbound_airline          0
outbound_flight_number    0
outbound_departure        0
outbound_arrival          0
outbound_duration         0
outbound_stops            0
return_airline            0
return_flight_number      0
return_departure          0
return_arrival            0
return_duration           0
return_stops              0
total_duration            0
price                     0
currency                  0
travel_class              0
dtype: int64

Duplicate records: 9

Records per destination:
destination
BLR    13
BOM    17
COK    18
DEL    29
GOI     4
JAI     3
SXR     3
UDR     3
dtype: int64


In [49]:
# Save the cleaned flight dataset

output_path = "../data/cleaned/flights_data.csv"

flights_master.to_csv(
    output_path,
    index=False
)

print(f"Saved {len(flights_master)} flight records.")
print(output_path)

Saved 90 flight records.
../data/cleaned/flights_data.csv
